# From Omilia Calls to a Training-Ready ML Pipeline in Snowflake

**Welcome to Day 2 of Omilia & Snowflake Open Doors — The Art of the Possible with Cortex AI.**

In this hands-on session you will build a complete **governed, ML-ready data pipeline** from raw call-center recordings to a registered model with full lineage — all inside Snowflake, with zero infrastructure to manage.

Here’s what you’ll do:

1. **Load** raw call-center data from a shared stage
2. **Govern** — anonymize transcripts (AI_REDACT) and tag columns for discoverability
3. **Annotate** — simulate human annotation write-back (AI_CLASSIFY) into Snowflake
4. **Snapshot** — create an immutable, versioned ML Dataset (a "recipe")
5. **Train** — train a model on the dataset and register it in the Model Registry
6. **Infer** — run batch inference and save predictions to prove end-to-end value
7. **Trace** — view ML Lineage in Horizon Catalog
8. **Query** — create a Cortex Agent for natural-language analytics

---

### Before you begin
Change the `MY_SCHEMA` variable in the next cell to your assigned name (e.g., `ALEJANDRO_P`).

In [ ]:
-- ======================================================================
-- STEP 0: SETUP
-- ======================================================================

-- >> CHANGE THIS to your assigned schema name (first name + initial) <<
SET MY_SCHEMA = 'YOURNAME_X';

USE ROLE WORKSHOP_PARTICIPANT_2026;
USE WAREHOUSE WORKSHOP_WH;
USE DATABASE OMILIA_WORKSHOP_DAY2;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER($MY_SCHEMA);
USE SCHEMA IDENTIFIER($MY_SCHEMA);

---
## Step 1: Load Raw Data

**The role:** Data Engineer at Omilia

**The challenge:** Raw production call records arrive from Omilia’s telephony platform. They contain customer names, emails, phone numbers embedded in transcripts — completely ungoverned.

**The solution:** Load them into Snowflake as-is. Governance comes next.

**Expected:** 200 rows loaded into `CALLS_RAW`.

In [ ]:
-- ======================================================================
-- STEP 1: Load raw call data
-- ======================================================================

CREATE OR REPLACE TABLE CALLS_RAW (
    CALL_ID VARCHAR,
    CUSTOMER_ID VARCHAR,
    CUSTOMER_NAME VARCHAR,
    CUSTOMER_EMAIL VARCHAR,
    QUEUE VARCHAR,
    LANG VARCHAR,
    CALL_START_TS TIMESTAMP_NTZ,
    CALL_END_TS TIMESTAMP_NTZ,
    DURATION_SEC INTEGER,
    AGENT_ID VARCHAR,
    CHANNEL VARCHAR,
    RAW_TRANSCRIPT TEXT
);

COPY INTO CALLS_RAW
FROM @OMILIA_WORKSHOP_DAY2.SHARED.WORKSHOP_STAGE/calls_raw.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"'
               ESCAPE_UNENCLOSED_FIELD = NONE);

SELECT COUNT(*) AS rows_loaded FROM CALLS_RAW;

In [ ]:
-- Preview: notice the PII (names, emails) in the transcripts
SELECT CALL_ID, CUSTOMER_NAME, CUSTOMER_EMAIL, QUEUE, LANG,
       LEFT(RAW_TRANSCRIPT, 120) AS transcript_preview
FROM CALLS_RAW
LIMIT 5;

---
## Step 2: Govern — Anonymize & Tag

**The role:** Data Governance Engineer

**The challenge:** Before this data can be used for ML training or analytics, two governance gates must pass:
1. **PII removal** — names, emails, phone numbers must be masked
2. **Column tagging** — columns must be classified so data stewards know what’s sensitive and policies can be enforced automatically

**The solution:** `AI_REDACT` for instant PII removal, then Snowflake Object Tags for metadata governance.

> **Why tags matter:** Once tagged, you can apply tag-based masking policies, audit who accessed sensitive columns, and ensure new tables inherit governance rules automatically.

**Expected:** `CALLS_ANON` table with PII removed and governance tags applied.

In [ ]:
-- ======================================================================
-- STEP 2A: Anonymize with AI_REDACT
-- ======================================================================

CREATE OR REPLACE TABLE CALLS_ANON AS
SELECT
    CALL_ID,
    CUSTOMER_ID,
    QUEUE,
    LANG,
    CALL_START_TS,
    CALL_END_TS,
    DURATION_SEC,
    AGENT_ID,
    CHANNEL,
    AI_REDACT(RAW_TRANSCRIPT) AS TRANSCRIPT_ANON
FROM CALLS_RAW;

In [ ]:
-- Compare: raw vs anonymized
SELECT
    r.CALL_ID,
    LEFT(r.RAW_TRANSCRIPT, 80) AS original,
    LEFT(a.TRANSCRIPT_ANON, 80) AS anonymized
FROM CALLS_RAW r
JOIN CALLS_ANON a ON r.CALL_ID = a.CALL_ID
LIMIT 3;

In [ ]:
-- ======================================================================
-- STEP 2B: Apply Governance Tags
-- ======================================================================
-- Tags make columns discoverable, auditable, and policy-ready.
-- These tags were pre-created by your admin in the SHARED schema.

ALTER TABLE CALLS_ANON
  MODIFY COLUMN CALL_ID SET TAG OMILIA_WORKSHOP_DAY2.SHARED.DATA_DOMAIN = 'call_center';

ALTER TABLE CALLS_ANON
  MODIFY COLUMN TRANSCRIPT_ANON SET TAG OMILIA_WORKSHOP_DAY2.SHARED.SENSITIVITY = 'internal';

ALTER TABLE CALLS_ANON
  MODIFY COLUMN CUSTOMER_ID SET TAG OMILIA_WORKSHOP_DAY2.SHARED.SENSITIVITY = 'confidential';

-- Verify: see all tags applied to this table
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('CALLS_ANON', 'TABLE'));

---
## Step 3: Annotation Write-Back

**The role:** ML Engineer + Annotation Team

**The challenge:** Your intent-classification model couldn’t confidently predict the intent for ~30% of calls. These need human review — an annotator listens to the audio and labels the correct intent.

**The solution:** In production, an external annotation UI (Label Studio, Prodigy, or Omilia’s internal labeling tool) writes human labels back into a Snowflake table via connector or API. Here we simulate that write-back using `AI_CLASSIFY` — this produces semantically meaningful labels based on transcript content (what a real annotator would label), making the downstream model training realistic.

The result: a `CALLS_TRAINING_READY` table where every row has either a human annotation or a model prediction, with a `LABEL_SOURCE` column tracking provenance.

**Expected:** ~65 rows with `LABEL_SOURCE = 'human'` (meaningful intent labels), rest with `'model'`.

In [ ]:
-- ======================================================================
-- STEP 3: Annotation Write-Back (simulated with AI_CLASSIFY)
-- ======================================================================
-- In production: an annotation UI (Label Studio, Prodigy, or Omilia's
-- internal tool) writes human labels back into this table via API.
-- Here we simulate realistic annotations using AI_CLASSIFY on the
-- transcript — producing content-based labels a human would assign.

CREATE OR REPLACE TABLE HUMAN_ANNOTATIONS AS
SELECT
    CALL_ID,
    AI_CLASSIFY(
        TRANSCRIPT_ANON::VARCHAR,
        ['billing_dispute', 'service_outage', 'plan_upgrade', 'device_issue', 'account_closure', 'general_inquiry']
    ):labels[0]::VARCHAR AS INTENT_LABEL,
    'annotator_' || MOD(ABS(HASH(CALL_ID)), 5) AS ANNOTATOR_ID,
    DATEADD(hour, -MOD(ABS(HASH(CALL_ID)), 48), CURRENT_TIMESTAMP()::TIMESTAMP_NTZ) AS ANNOTATED_AT,
    ROUND(0.85 + (MOD(ABS(HASH(CALL_ID)), 15) / 100.0), 2) AS CONFIDENCE
FROM CALLS_ANON
WHERE MOD(ABS(HASH(CALL_ID)), 3) = 0;

SELECT INTENT_LABEL, COUNT(*) AS cnt
FROM HUMAN_ANNOTATIONS
GROUP BY INTENT_LABEL ORDER BY cnt DESC;

In [ ]:
-- Merge: combine anonymized data with annotation labels
CREATE OR REPLACE TABLE CALLS_TRAINING_READY AS
SELECT
    a.*,
    h.INTENT_LABEL,
    h.ANNOTATOR_ID,
    h.ANNOTATED_AT,
    h.CONFIDENCE AS ANNOTATION_CONFIDENCE,
    CASE WHEN h.CALL_ID IS NOT NULL THEN 'human' ELSE 'model' END AS LABEL_SOURCE
FROM CALLS_ANON a
LEFT JOIN HUMAN_ANNOTATIONS h ON a.CALL_ID = h.CALL_ID;

-- Verify: count by label source
SELECT LABEL_SOURCE, COUNT(*) AS cnt
FROM CALLS_TRAINING_READY
GROUP BY LABEL_SOURCE;

---
## Step 4: Create an Immutable ML Dataset

**The role:** ML Engineer

**The challenge:** You need reproducibility. If you retrain a model 6 months from now, you must be able to prove *exactly* which data was used. Tables change — datasets don’t.

**The solution:** Snowflake ML Datasets. Each version is an **immutable, point-in-time snapshot** — a frozen "recipe" of features, labels, and filters.

The whole point: your team can have multiple datasets (recipes):
- **v1** = all human-annotated calls (what we’re creating now)
- **v2** = adds model-predicted labels for unannotated rows
- **v3** = Greek-only calls for a language-specific model
- **v4** = filtered to billing queue for a specialized classifier

Each version is traceable, reproducible, and shows up in ML Lineage.

**Expected:** Dataset `CALLS_TRAINING_DATASET` with version `v1`.

In [ ]:
# ======================================================================
# STEP 4: Create Immutable ML Dataset
# ======================================================================
# Using the Python API ensures ML Lineage captures the link
# from the source table (CALLS_TRAINING_READY) to the dataset.
# SQL-based dataset creation does NOT record this lineage.

from snowflake.snowpark.context import get_active_session
from snowflake.ml import dataset

session = get_active_session()

# Read training-ready data as a Snowpark DataFrame
source_df = session.table("CALLS_TRAINING_READY")

# Create the dataset with version v1 — this captures table → dataset lineage
ds = dataset.create_from_dataframe(
    session,
    name="CALLS_TRAINING_DATASET",
    version="v1",
    input_dataframe=source_df,
)

print(f"Dataset: {ds.fully_qualified_name}")
print(f"Version: {ds.selected_version.name}")
print(f"Created: {ds.selected_version.created_on}")

---
## Step 5: Train & Register a Model

**The role:** Data Scientist

**The challenge:** Train an intent classifier on the annotated data and register it so it’s versioned, discoverable, and traceable back to the exact dataset that produced it.

**The solution:** Snowpark ML for training + Model Registry for registration. This completes the lineage chain: Source Table → Dataset → Model.

> **Note:** This is a minimal example to show the pattern. As we discussed in Day 1, in production you’d leverage the full Snowflake MLOps stack:
> - **Feature Store** for managed feature pipelines
> - **Online Feature Store** for real-time serving
> - **Experiments** for hyperparameter tracking
> - **Model Explainability** (SHAP) for interpretability
> - **ML Observability** for drift detection
> - **Model Registry** with full versioning and deployment workflows

**Expected:** Model `INTENT_CLASSIFIER` version `v1` in the Model Registry.

In [ ]:
# ======================================================================
# STEP 5: Train & Register Model
# ======================================================================
from snowflake.snowpark.context import get_active_session
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.registry import Registry
from snowflake.ml.dataset import load_dataset

session = get_active_session()

# Load from the immutable dataset — the production pattern
ds = load_dataset(session, "CALLS_TRAINING_DATASET", "v1")
train_df = ds.read.to_snowpark_dataframe()

# Only use human-annotated rows for training
labeled_df = train_df.filter(train_df["LABEL_SOURCE"] == "human")
print(f"Training on {labeled_df.count()} human-annotated rows")

# Train a simple intent classifier
model = RandomForestClassifier(
    n_estimators=50,
    input_cols=["DURATION_SEC"],
    label_cols=["INTENT_LABEL"],
)
model.fit(labeled_df)

# Register model in the Model Registry
reg = Registry(session=session)
mv = reg.log_model(
    model,
    model_name="INTENT_CLASSIFIER",
    version_name="v1",
    sample_input_data=labeled_df.select("DURATION_SEC").limit(10),
    comment="Intent classifier trained on human-annotated call data (Day 2 workshop)"
)
print(f"Model registered: {mv.model_name} version {mv.version_name}")

In [ ]:
-- Verify: model appears in the registry
SHOW MODELS;

---
## Step 6: Batch Inference — Prove It Works

**The role:** ML Engineer

**The challenge:** The model is registered — but how do you prove the end-to-end pipeline delivers value? Run batch inference on the full dataset, save predictions to a table, and compare predicted vs actual labels.

**The solution:** Call the registered model directly from SQL using the `MODEL!PREDICT()` pattern. This saves predictions alongside the original data so you can measure accuracy and detect drift.

**Expected:** `CALLS_PREDICTIONS` table with predicted intent labels for all rows.

In [ ]:
-- ======================================================================
-- STEP 6: Batch Inference
-- ======================================================================
-- Run the registered model on all training-ready calls to generate predictions.

CREATE OR REPLACE TABLE CALLS_PREDICTIONS AS
SELECT
    CALL_ID, QUEUE, LANG, DURATION_SEC, CHANNEL, INTENT_LABEL, LABEL_SOURCE,
    INTENT_CLASSIFIER!PREDICT(DURATION_SEC):OUTPUT_INTENT_LABEL::VARCHAR AS PREDICTED_INTENT_LABEL
FROM CALLS_TRAINING_READY;

-- Compare predictions vs actual labels for human-annotated rows
SELECT
    CALL_ID, QUEUE, DURATION_SEC,
    INTENT_LABEL AS ACTUAL_LABEL,
    PREDICTED_INTENT_LABEL,
    CASE WHEN INTENT_LABEL = PREDICTED_INTENT_LABEL THEN 'correct' ELSE 'mismatch' END AS MATCH,
    LABEL_SOURCE
FROM CALLS_PREDICTIONS
WHERE LABEL_SOURCE = 'human'
LIMIT 15;

---
## Step 6: View ML Lineage

**The role:** Data Governance Officer / ML Engineer

**The challenge:** Auditors ask: "Which data trained this model? Was PII removed? Who annotated the labels?" You need to answer instantly.

**The solution:** Snowflake ML Lineage in Horizon Catalog. It automatically traces the full chain:

```
CALLS_TRAINING_READY (table) → CALLS_TRAINING_DATASET v1 (dataset) → INTENT_CLASSIFIER v1 (model)
```

### Try it now:

1. In Snowsight, go to **Catalog → Database Explorer**
2. Navigate to your schema → **Models** → `INTENT_CLASSIFIER`
3. Click on version **v1** → select the **Lineage** tab
4. You should see the full graph: source table → dataset → model

> This is governance in action — every model is traceable back to its exact training data, with full provenance.

---
## Step 7: Cortex Agent — Natural Language Analytics

**The role:** Business Analyst / Operations Lead

**The challenge:** You have governed, annotated, training-ready data — but analysts shouldn’t need SQL to get insights. They want to ask questions in natural language.

**The solution:** Create a **Semantic View** (business-friendly data model) and a **Cortex Agent** that translates natural language to SQL.

**Expected:** An agent you can chat with in Snowflake CoWork.

In [ ]:
-- ======================================================================
-- STEP 8A: Create a Semantic View (with verified queries)
-- ======================================================================

CREATE OR REPLACE SEMANTIC VIEW CALLS_SEMANTIC_VIEW

  TABLES (
    calls AS CALLS_TRAINING_READY
      PRIMARY KEY (CALL_ID)
      COMMENT = 'Omilia call center training-ready data with human and model annotations'
  )

  DIMENSIONS (
    calls.queue AS QUEUE
      WITH SYNONYMS = ('department', 'team')
      COMMENT = 'Call queue: billing, technical_support, cancellations, sales, general',
    calls.lang AS LANG
      WITH SYNONYMS = ('language', 'locale')
      COMMENT = 'Language of the call: en (English), el (Greek), es (Spanish)',
    calls.channel AS CHANNEL
      COMMENT = 'Communication channel: phone, virtual_assistant, chat',
    calls.agent_id AS AGENT_ID
      WITH SYNONYMS = ('agent', 'representative')
      COMMENT = 'Agent who handled the call',
    calls.call_start_ts AS CALL_START_TS
      COMMENT = 'Call start timestamp',
    calls.intent_label AS INTENT_LABEL
      WITH SYNONYMS = ('intent', 'category', 'reason for calling')
      COMMENT = 'Annotated intent: billing_dispute, service_outage, plan_upgrade, device_issue, account_closure, general_inquiry',
    calls.label_source AS LABEL_SOURCE
      COMMENT = 'Who provided the label: human (annotator) or model (AI-predicted)'
  )

  METRICS (
    calls.total_calls AS COUNT(CALL_ID)
      COMMENT = 'Total number of calls',
    calls.avg_duration AS AVG(DURATION_SEC)
      WITH SYNONYMS = ('average call length', 'mean duration')
      COMMENT = 'Average call duration in seconds',
    calls.human_annotated_count AS COUNT(CASE WHEN LABEL_SOURCE = 'human' THEN 1 END)
      COMMENT = 'Number of calls with human annotations',
    calls.annotation_coverage AS AVG(CASE WHEN LABEL_SOURCE = 'human' THEN 1.0 ELSE 0.0 END) * 100
      WITH SYNONYMS = ('coverage rate', 'labeling progress')
      COMMENT = 'Percentage of calls with human annotations (0-100)',
    calls.max_duration AS MAX(DURATION_SEC)
      COMMENT = 'Longest call duration in seconds',
    calls.min_duration AS MIN(DURATION_SEC)
      COMMENT = 'Shortest call duration in seconds'
  )

  COMMENT = 'Semantic view over Omilia annotated call data for natural-language analytics'

  AI_SQL_GENERATION 'When showing percentages, round to 1 decimal place. When showing durations, display in seconds. Always order results by the most relevant metric descending unless the user specifies otherwise.'

  AI_VERIFIED_QUERIES (
    calls_by_queue AS (
      QUESTION 'How many calls are in each queue?'
      VERIFIED_AT 1750780800
      ONBOARDING_QUESTION TRUE
      VERIFIED_BY '(STEWARD = workshop_admin)'
      SQL 'SELECT calls.QUEUE, COUNT(calls.CALL_ID) AS TOTAL_CALLS FROM CALLS_TRAINING_READY AS calls GROUP BY calls.QUEUE ORDER BY TOTAL_CALLS DESC'
    ),
    intent_distribution AS (
      QUESTION 'What is the distribution of call intents for human-annotated calls?'
      VERIFIED_AT 1750780800
      ONBOARDING_QUESTION TRUE
      VERIFIED_BY '(STEWARD = workshop_admin)'
      SQL 'SELECT calls.INTENT_LABEL, COUNT(calls.CALL_ID) AS TOTAL_CALLS FROM CALLS_TRAINING_READY AS calls WHERE calls.LABEL_SOURCE = ''human'' GROUP BY calls.INTENT_LABEL ORDER BY TOTAL_CALLS DESC'
    ),
    avg_duration_by_queue AS (
      QUESTION 'What is the average call duration per queue?'
      VERIFIED_AT 1750780800
      ONBOARDING_QUESTION TRUE
      VERIFIED_BY '(STEWARD = workshop_admin)'
      SQL 'SELECT calls.QUEUE, AVG(calls.DURATION_SEC) AS AVG_DURATION FROM CALLS_TRAINING_READY AS calls GROUP BY calls.QUEUE ORDER BY AVG_DURATION DESC'
    ),
    annotation_coverage AS (
      QUESTION 'What percentage of calls have human annotations?'
      VERIFIED_AT 1750780800
      ONBOARDING_QUESTION FALSE
      VERIFIED_BY '(STEWARD = workshop_admin)'
      SQL 'SELECT ROUND(AVG(CASE WHEN calls.LABEL_SOURCE = ''human'' THEN 1.0 ELSE 0.0 END) * 100, 1) AS ANNOTATION_COVERAGE_PCT FROM CALLS_TRAINING_READY AS calls'
    ),
    calls_by_language AS (
      QUESTION 'How many calls per language?'
      VERIFIED_AT 1750780800
      ONBOARDING_QUESTION FALSE
      VERIFIED_BY '(STEWARD = workshop_admin)'
      SQL 'SELECT calls.LANG, COUNT(calls.CALL_ID) AS TOTAL_CALLS FROM CALLS_TRAINING_READY AS calls GROUP BY calls.LANG ORDER BY TOTAL_CALLS DESC'
    )
  );

In [ ]:
-- ======================================================================
-- STEP 8B: Create a Cortex Agent (with sample questions)
-- ======================================================================

CREATE OR REPLACE AGENT CALLS_ANALYST_AGENT
  COMMENT = 'Omilia Call Center Analyst — ask questions about call volumes, intents, annotation coverage, and queue performance'
  FROM SPECIFICATION
  $$
  instructions:
    response: |
      You are the Omilia Call Center Analyst. Answer questions about call center operations clearly and concisely.
      When showing data with categories or comparisons, always include a chart visualization.
      Use bar charts for category comparisons and line charts for time trends.
      Format numbers with appropriate precision (percentages to 1 decimal, durations in seconds).
      When asked about model performance, compare predicted vs actual labels.
    sample_questions:
      - question: "How many calls are in each queue? Show a bar chart."
      - question: "What is the distribution of intents for human-annotated calls?"
      - question: "What is the average call duration per queue?"
      - question: "What percentage of calls have human annotations?"
      - question: "Which language has the most calls?"
      - question: "Show me the top 3 queues with the longest average duration."

  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "CallsAnalyst"
        description: "Queries Omilia call center data including call volumes by queue, intent distribution, annotation coverage, average duration, channel breakdowns, and language statistics."

  tool_resources:
    CallsAnalyst:
      semantic_view: "CALLS_SEMANTIC_VIEW"
      execution_environment:
        type: "warehouse"
        warehouse: "WORKSHOP_WH"
  $$;

### Test your Agent

1. Open **Snowflake CoWork** (top-right corner of Snowsight → CoWork icon)
2. Select your **CALLS_ANALYST_AGENT** from the agent list
3. Ask it a question, for example:

```
What are the top intent categories by call volume? Show a chart.
```

```
What percentage of calls have human annotations vs model predictions?
```

```
Which queue has the longest average call duration?
```

The agent uses the Semantic View you created to translate natural language into SQL and return results with charts.

---

### What you built today

| Layer | Object | Purpose |
|-------|--------|---------|
| Raw | `CALLS_RAW` | Production data as-is |
| Governed | `CALLS_ANON` (tagged) | PII removed, columns tagged for policy enforcement |
| Annotated | `CALLS_TRAINING_READY` | Human labels merged back from annotation tool |
| Dataset | `CALLS_TRAINING_DATASET v1` | Immutable snapshot ("recipe") for reproducibility |
| Model | `INTENT_CLASSIFIER v1` | Trained + registered with full lineage |
| Agent | `CALLS_ANALYST_AGENT` | Natural-language analytics over governed data |

This is the **ML-ready data pipeline** pattern:
- Production data flows in
- Gets governed (anonymized + tagged)
- Human annotations flow back
- Immutable datasets freeze the recipe
- Models train and register with lineage
- Analysts query everything via natural language

---
*Workshop created for Omilia Open Doors Day 2 | Powered by Snowflake*

---
## Bonus Hackathon: What Can Your Team Build?

Open **Cortex Code** and prompt it to build something that solves a real ML challenge for your team using what you’ve learned today.

You can ask Cortex Code to:
- Create a new schema for your hackathon work (`YOURNAME_HACKATHON`)
- Generate mock data relevant to your team’s domain
- Build annotation pipelines, classifiers, or monitoring dashboards
- Train and register models with full lineage

**This is your time to explore the future of ML for your team with Snowflake.**

---

### Sample prompts to adapt

**1. Build your own classifier:**
```
In schema OMILIA_WORKSHOP_DAY2.<MY_SCHEMA>, create mock data for
<YOUR_TEAM'S_CLASSIFICATION_TASK> (e.g., ticket priority, churn risk,
lead scoring) with at least 200 rows. Train a classifier using Snowpark ML,
register it in the Model Registry, and verify lineage in Horizon Catalog.
```

**2. Annotation pipeline for your domain:**
```
Build an annotation write-back pipeline for <YOUR_TEAM'S_LABELING_NEED>.
Create a queue table of unlabeled items, simulate human annotations,
merge them back, and create an immutable Dataset version. Show the
before/after in the lineage graph.
```

**3. ML monitoring dashboard:**
```
Using the model I registered (INTENT_CLASSIFIER), run batch inference on
OMILIA_WORKSHOP_DAY2.<MY_SCHEMA>.CALLS_TRAINING_READY and create a
Streamlit app that shows prediction distribution, confidence scores,
and flags low-confidence predictions that need human review.
Should run on my warehouse WORKSHOP_WH.
```

**4. Anomaly detection on your metrics:**
```
In schema OMILIA_WORKSHOP_DAY2.<MY_SCHEMA>, create mock time-series
data for <YOUR_TEAM'S_METRIC> (e.g., daily call volume, hourly error
rate, weekly churn) with 100+ rows. Train an Anomaly Detection model,
save results to a table, and build a Streamlit app to navigate the
anomalies. Should run on my warehouse WORKSHOP_WH.
```

---
> After 15 minutes, be ready to come on stage and show what you built in 60 seconds!